In [1]:
import os
import re
import datetime as _dt
import glob as glob_module
import pandas as pd
import openpyxl
from openpyxl.utils import column_index_from_string, get_column_letter
from IPython.display import display

# ── Path Setup ─────────────────────────────────────────────────────────────────
first_glob = os.path.expanduser('~').replace('\\', '/')
test_path  = f'{first_glob}/Concentrix Corporation'
if not os.path.exists(test_path):
    raise FileNotFoundError(f'Not found the path: {test_path}')

folder_paths = {
    'input_ou_mail' : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_OU_MAIL',
    'output_ou_mail': f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ou_email.xlsx',
}
output_ou_mail = folder_paths['output_ou_mail']

# ── Config ─────────────────────────────────────────────────────────────────────
ROW_START = 5
ROW_END   = 53

LC_SHEET_CONFIG = {
    'VNM - OU'  : ('E', 'Y'),
    'Kol - OU'  : ('E', 'Y'),
    'Pune - OU' : ('E', 'Y'),
    'Gobal - OU': ('G', 'AA'),
}
NL_SHEET_CONFIG = {
    'VNM - OU'  : ('E', 'Y'),
    'Kol - OU'  : ('E', 'Y'),
    'Pune - OU' : ('E', 'Y'),
    'Gobal - OU': ('F', 'Z'),
}

DAYS            = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
METRICS         = ['Req W', 'Prov', 'DIFF']
DAY_METRIC_COLS = [f'{d}_{m}' for d in DAYS for m in METRICS]
DAY_OFFSET      = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}

TZ_OFFSETS = {
    'VNT': pd.Timedelta(hours=15),
    'IST': pd.Timedelta(hours=13, minutes=30),
    'CLT': pd.Timedelta(hours=4),
}

OU_TO_SITE = {
    'VNM - OU'    : 'Concentrix (Ho Chi Minh City)',
    'Kol - OU'    : 'Concentrix (Kolkata)',
    'Pune - OU'   : 'Concentrix (Pune)',
    'Gobal - OU'  : 'Concentrix (Global)',
    'Global - OU' : 'Concentrix (Global)',
    'Egypt - OU'  : 'Concentrix (Cairo)',
}

# ── Helpers ────────────────────────────────────────────────────────────────────
def find_latest_file(folder: str, keyword: str) -> str:
    matches = glob_module.glob(os.path.join(folder, f'*{keyword}*.xlsx'))
    if not matches:
        raise FileNotFoundError(f'No file matched "{keyword}" in: {folder}')
    return max(matches, key=os.path.getmtime)

def extract_week_start(file_path: str) -> pd.Timestamp:
    match = re.search(r'(\d{2})_(\d{2})_(\d{2})', os.path.basename(file_path))
    if not match:
        return pd.NaT
    month, day, year = int(match[1]), int(match[2]), int(match[3]) + 2000
    return pd.Timestamp(year=year, month=month, day=day)

def _resolve_sheet(sheetnames: list, target: str) -> str | None:
    if target in sheetnames:
        return target
    def _norm(s): return s.lower().replace(' ', '').replace('-', '')
    t_norm = _norm(target)
    for s in sheetnames:
        if _norm(s) == t_norm:
            return s
    t_loose = t_norm.replace('global', 'gobal')
    for s in sheetnames:
        if _norm(s).replace('global', 'gobal') == t_loose:
            return s
    return None

def _map_site(ou_val: str) -> str:
    if ou_val in OU_TO_SITE:
        return OU_TO_SITE[ou_val]
    norm = lambda s: s.lower().replace(' ', '').replace('-', '')
    for k, v in OU_TO_SITE.items():
        if norm(k).replace('global', 'gobal') == norm(ou_val).replace('global', 'gobal'):
            return v
    return ou_val

def _to_time_str(val) -> str | None:
    """Handle datetime.time, datetime.datetime, hoặc fallback pd.to_datetime."""
    if val is None:
        return None
    if isinstance(val, _dt.time):
        return val.strftime('%H:%M')
    if isinstance(val, _dt.datetime):
        return val.strftime('%H:%M')
    ts = pd.to_datetime(val, errors='coerce')
    return None if pd.isna(ts) else ts.strftime('%H:%M')

def read_ou_sheet(ws, col_start: str, col_end: str,
                  label_col: str = 'A',
                  row_start: int = ROW_START,
                  row_end: int   = ROW_END) -> pd.DataFrame:
    lbl_idx = column_index_from_string(label_col)
    d_start = column_index_from_string(col_start)
    d_end   = column_index_from_string(col_end)
    records = []
    for row in range(row_start, row_end + 1):
        label = ws.cell(row=row, column=lbl_idx).value
        vals  = [ws.cell(row=row, column=c).value for c in range(d_start, d_end + 1)]
        records.append([label] + vals)
    tmp_cols = ['PST'] + [get_column_letter(c) for c in range(d_start, d_end + 1)]
    df = pd.DataFrame(records, columns=tmp_cols)
    df = df[df['PST'].astype(str).str.strip() != 'PST'].reset_index(drop=True)
    df['PST'] = df['PST'].apply(_to_time_str)
    data_cols  = [c for c in df.columns if c != 'PST']
    rename_map = {old: new for old, new in zip(data_cols, DAY_METRIC_COLS[:len(data_cols)])}
    df.rename(columns=rename_map, inplace=True)
    return df

def load_ou_file(file_path: str, sheet_config: dict, file_label: str) -> dict:
    wb      = openpyxl.load_workbook(file_path, data_only=True)
    week_dt = extract_week_start(file_path)
    print(f'\n📗 {os.path.basename(file_path)}  |  Week start: {week_dt.date()}')
    print(f'   Sheets: {wb.sheetnames}')
    result = {}
    for target, (col_s, col_e) in sheet_config.items():
        actual = _resolve_sheet(wb.sheetnames, target)
        if actual is None:
            print(f'   ⚠️  Sheet not found: "{target}" — skipped')
            continue
        df = read_ou_sheet(wb[actual], col_s, col_e)
        df.insert(0, 'Week', week_dt)
        df.insert(0, 'File', file_label)
        df.insert(0, 'Site', _map_site(actual))
        result[actual] = df
        print(f'   ✅ "{actual}" → {df.shape}')
    wb.close()
    return result

# ── Load ───────────────────────────────────────────────────────────────────────
lc_path = find_latest_file(folder_paths['input_ou_mail'], 'Global OU LC Chat')
nl_path = find_latest_file(folder_paths['input_ou_mail'], 'Global OU NL Chat')

lc_data = load_ou_file(lc_path, LC_SHEET_CONFIG, file_label='Lodging chat')
nl_data = load_ou_file(nl_path, NL_SHEET_CONFIG, file_label='Non-Lodging chat')

# ── Combine & Melt ─────────────────────────────────────────────────────────────
all_combined = pd.concat(
    [*lc_data.values(), *nl_data.values()],
    ignore_index=True
)

id_cols  = ['Site', 'File', 'Week', 'PST']
val_cols = [c for c in all_combined.columns if c not in id_cols]

df_long = all_combined.melt(
    id_vars    = id_cols,
    value_vars = val_cols,
    var_name   = '_col',
    value_name = 'Value'
)

df_long[['Day', 'OU Status']] = df_long['_col'].str.split('_', n=1, expand=True)

df_long['PST Date'] = (
    df_long['Week'] +
    df_long['Day'].map(DAY_OFFSET).apply(lambda x: pd.Timedelta(days=x))
)

df_long['PST Datetime'] = pd.to_datetime(
    df_long['PST Date'].dt.strftime('%Y-%m-%d') + ' ' + df_long['PST'].astype(str),
    errors='coerce'
)

for tz, offset in TZ_OFFSETS.items():
    df_long[f'{tz} Datetime'] = df_long['PST Datetime'] + offset
    df_long[f'{tz} Date']     = df_long[f'{tz} Datetime'].dt.normalize()

df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce').round(2)

df_long['VNT'] = df_long['VNT Datetime'].dt.strftime('%H:%M')

df_long = (
    df_long
    .rename(columns={'File': 'LOB'})
    .drop(columns=['_col', 'Day'])
    .reindex(columns=[
        'Site', 'LOB', 'Week',
        'PST Date', 'PST', 'PST Datetime',
        'VNT Datetime', 'VNT Date', 'VNT',
        'IST Datetime', 'IST Date',
        'CLT Datetime', 'CLT Date',
        'OU Status', 'Value',
    ])
    .reset_index(drop=True)
)

# ── Export ─────────────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(output_ou_mail), exist_ok=True)
df_long.to_excel(output_ou_mail, index=False)

print(f'\nLong format : {df_long.shape}')
print(f'Columns     : {df_long.columns.tolist()}')
display(df_long.head(21))

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)



📗 Global OU LC Chat 07_20_26.xlsx  |  Week start: 2026-07-20
   Sheets: ['LookingGlass Template', 'Client View', 'Forecast var.', 'Week 1 - OU', 'Weekly Staffing - Week 1', 'Email Template - Week 1', 'Performance Analysis', 'Egypt - OU', 'Egypt Staffing - Data', 'VNM - OU ', 'VNM Staffing - Data', 'Kol - OU ', 'Kol Staffing - Data', 'Pune - OU', 'Pune Staffing - Data', 'Gobal - OU', 'Volume']
   ✅ "VNM - OU " → (48, 25)
   ✅ "Kol - OU " → (48, 25)
   ✅ "Pune - OU" → (48, 25)
   ✅ "Gobal - OU" → (48, 25)


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)



📗 Global OU NL Chat 07_20_26.xlsx  |  Week start: 2026-07-20
   Sheets: ['LookingGlass Template', 'Client View', 'Forecast var.', 'Week 1 - OU', 'Weekly Staffing - Week 1', 'Email Template - Week 1', 'Performance Analysis', 'Egypt - OU', 'VNM - OU ', 'VNM Staffing - Data', 'Egypt Staffing - Data', 'Kol Staffing - Data', 'Pune - OU', 'Sheet1', 'Pune Staffing - Data', 'Kol - OU ', 'Gobal - OU', 'Requirement Trend', 'Req+Open', 'Volume', 'Shinkage', 'Net Staffing']
   ✅ "VNM - OU " → (48, 25)
   ✅ "Kol - OU " → (48, 25)
   ✅ "Pune - OU" → (48, 25)
   ✅ "Gobal - OU" → (48, 25)

Long format : (8064, 15)
Columns     : ['Site', 'LOB', 'Week', 'PST Date', 'PST', 'PST Datetime', 'VNT Datetime', 'VNT Date', 'VNT', 'IST Datetime', 'IST Date', 'CLT Datetime', 'CLT Date', 'OU Status', 'Value']


,Site,LOB,Week,PST Date,PST,PST Datetime,VNT Datetime,VNT Date,VNT,IST Datetime,IST Date,CLT Datetime,CLT Date,OU Status,Value
0,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,00:00,2026-07-20 00:00:00,2026-07-20 15:00:00,2026-07-20,15:00,2026-07-20 13:30:00,2026-07-20,2026-07-20 04:00:00,2026-07-20,Req W,11.40
1,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,00:30,2026-07-20 00:30:00,2026-07-20 15:30:00,2026-07-20,15:30,2026-07-20 14:00:00,2026-07-20,2026-07-20 04:30:00,2026-07-20,Req W,11.73
2,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,01:00,2026-07-20 01:00:00,2026-07-20 16:00:00,2026-07-20,16:00,2026-07-20 14:30:00,2026-07-20,2026-07-20 05:00:00,2026-07-20,Req W,8.26
3,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,01:30,2026-07-20 01:30:00,2026-07-20 16:30:00,2026-07-20,16:30,2026-07-20 15:00:00,2026-07-20,2026-07-20 05:30:00,2026-07-20,Req W,7.40
4,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,02:00,2026-07-20 02:00:00,2026-07-20 17:00:00,2026-07-20,17:00,2026-07-20 15:30:00,2026-07-20,2026-07-20 06:00:00,2026-07-20,Req W,7.23
5,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,02:30,2026-07-20 02:30:00,2026-07-20 17:30:00,2026-07-20,17:30,2026-07-20 16:00:00,2026-07-20,2026-07-20 06:30:00,2026-07-20,Req W,6.15
6,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,03:00,2026-07-20 03:00:00,2026-07-20 18:00:00,2026-07-20,18:00,2026-07-20 16:30:00,2026-07-20,2026-07-20 07:00:00,2026-07-20,Req W,5.51
7,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,03:30,2026-07-20 03:30:00,2026-07-20 18:30:00,2026-07-20,18:30,2026-07-20 17:00:00,2026-07-20,2026-07-20 07:30:00,2026-07-20,Req W,9.79
8,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,04:00,2026-07-20 04:00:00,2026-07-20 19:00:00,2026-07-20,19:00,2026-07-20 17:30:00,2026-07-20,2026-07-20 08:00:00,2026-07-20,Req W,4.21
9,Concentrix (Ho Chi Minh City),Lodging chat,2026-07-20,2026-07-20,04:30,2026-07-20 04:30:00,2026-07-20 19:30:00,2026-07-20,19:30,2026-07-20 18:00:00,2026-07-20,2026-07-20 08:30:00,2026-07-20,Req W,6.21


In [2]:
import os
import re
import datetime as _dt
import glob as glob_module
import polars as pl
import openpyxl
from openpyxl.utils import column_index_from_string, get_column_letter
from IPython.display import display

# ── Path Setup ─────────────────────────────────────────────────────────────────
first_glob = os.path.expanduser('~').replace('\\', '/')
test_path  = f'{first_glob}/Concentrix Corporation'
if not os.path.exists(test_path):
    raise FileNotFoundError(f'Not found the path: {test_path}')

folder_paths = {
    'input_ou_mail'       : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_OU_MAIL',
    'output_ou_mail'      : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources/ou_email.xlsx',
    'iex_intervals_output': f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/OUTPUT_AGENT_IEX_INTERVALS',
    'hc_extend_by_month'  : f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month',
}
output_ou_mail = folder_paths['output_ou_mail']

# ── Config ─────────────────────────────────────────────────────────────────────
ROW_START = 5
ROW_END   = 53

LC_SHEET_CONFIG = {
    'VNM - OU'  : ('E', 'Y'),
    'Kol - OU'  : ('E', 'Y'),
    'Pune - OU' : ('E', 'Y'),
    'Gobal - OU': ('G', 'AA'),
}
NL_SHEET_CONFIG = {
    'VNM - OU'  : ('E', 'Y'),
    'Kol - OU'  : ('E', 'Y'),
    'Pune - OU' : ('E', 'Y'),
    'Gobal - OU': ('F', 'Z'),
}

DAYS            = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
METRICS         = ['Req W', 'Prov', 'DIFF']
DAY_METRIC_COLS = [f'{d}_{m}' for d in DAYS for m in METRICS]
DAY_OFFSET      = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}

TZ_OFFSETS = {
    'VNT': _dt.timedelta(hours=15),
    'IST': _dt.timedelta(hours=13, minutes=30),
    'CLT': _dt.timedelta(hours=4),
}

OU_TO_SITE = {
    'VNM - OU'    : 'Concentrix (Ho Chi Minh City)',
    'Kol - OU'    : 'Concentrix (Kolkata)',
    'Pune - OU'   : 'Concentrix (Pune)',
    'Gobal - OU'  : 'Concentrix (Global)',
    'Global - OU' : 'Concentrix (Global)',
    'Egypt - OU'  : 'Concentrix (Cairo)',
}

LOB_MAPPING = {
    'GEN_GEN_EN_GCS_GLG_CHT'            : 'Lodging chat',
    'GEN_GEN_EN_GCS_GNL_CHT'            : 'Non-Lodging chat',
    'GEN_GEN_EN_GCS_GLG_CHT_Concentrix' : 'Lodging chat',
    'GEN_GEN_EN_GCS_GNL_CHT_Concentrix' : 'Non-Lodging chat',
    'Lodging'                            : 'Lodging chat',
    'Lodging_Nesting'                    : 'Lodging chat',
    'Non_Lodging'                        : 'Non-Lodging chat',
}

LEAVE_LIST = ['Unscheduled', 'PTO', 'Termination', 'Offline', 'Paid Leave']

# ── Helpers ────────────────────────────────────────────────────────────────────
def find_latest_file(folder: str, keyword: str) -> str:
    matches = glob_module.glob(os.path.join(folder, f'*{keyword}*.xlsx'))
    if not matches:
        raise FileNotFoundError(f'No file matched "{keyword}" in: {folder}')
    return max(matches, key=os.path.getmtime)

def extract_week_start(file_path: str) -> _dt.date | None:
    match = re.search(r'(\d{2})_(\d{2})_(\d{2})', os.path.basename(file_path))
    if not match:
        return None
    month, day, year = int(match[1]), int(match[2]), int(match[3]) + 2000
    return _dt.date(year, month, day)

def _resolve_sheet(sheetnames: list, target: str) -> str | None:
    if target in sheetnames:
        return target
    def _norm(s): return s.lower().replace(' ', '').replace('-', '')
    t_norm = _norm(target)
    for s in sheetnames:
        if _norm(s) == t_norm:
            return s
    t_loose = t_norm.replace('global', 'gobal')
    for s in sheetnames:
        if _norm(s).replace('global', 'gobal') == t_loose:
            return s
    return None

def _map_site(ou_val: str) -> str:
    if ou_val in OU_TO_SITE:
        return OU_TO_SITE[ou_val]
    norm = lambda s: s.lower().replace(' ', '').replace('-', '')
    for k, v in OU_TO_SITE.items():
        if norm(k).replace('global', 'gobal') == norm(ou_val).replace('global', 'gobal'):
            return v
    return ou_val

def _to_time_str(val) -> str | None:
    if val is None:
        return None
    if isinstance(val, _dt.time):
        return val.strftime('%H:%M')
    if isinstance(val, _dt.datetime):
        return val.strftime('%H:%M')
    if isinstance(val, str):
        for fmt in ('%H:%M:%S', '%H:%M'):
            try:
                return _dt.datetime.strptime(val.strip(), fmt).strftime('%H:%M')
            except ValueError:
                continue
        return val
    return None

def input_data(folder_path: str) -> pl.DataFrame:
    file_paths = glob_module.glob(f'{folder_path}/*.xlsx') + glob_module.glob(f'{folder_path}/*.csv')
    df_list = []
    for file in file_paths:
        basename = os.path.basename(file)
        match = re.match(r'^(\d{4})', basename)
        if match and int(match.group(1)) < 2026:
            continue
        if file.endswith('.xlsx'):
            df = pl.read_excel(file)
        elif file.endswith('.csv'):
            try:
                df = pl.read_csv(file, encoding='utf-8')
            except Exception:
                df = pl.read_csv(file, encoding='ISO-8859-1', ignore_errors=True)
        df = df.with_columns(pl.all().cast(pl.String))
        df_list.append(df)
    return pl.concat(df_list, how='vertical') if df_list else pl.DataFrame()

def read_ou_sheet(ws, col_start: str, col_end: str,
                  label_col: str = 'A',
                  row_start: int = ROW_START,
                  row_end: int   = ROW_END) -> pl.DataFrame:
    lbl_idx       = column_index_from_string(label_col)
    d_start       = column_index_from_string(col_start)
    d_end         = column_index_from_string(col_end)
    tmp_data_cols = [get_column_letter(c) for c in range(d_start, d_end + 1)]
    pst_list      = []
    data_lists    = {col: [] for col in tmp_data_cols}
    for row in range(row_start, row_end + 1):
        pst_list.append(_to_time_str(ws.cell(row=row, column=lbl_idx).value))
        for i, c in enumerate(range(d_start, d_end + 1)):
            v = ws.cell(row=row, column=c).value
            try:
                data_lists[tmp_data_cols[i]].append(float(v) if v is not None else None)
            except (TypeError, ValueError):
                data_lists[tmp_data_cols[i]].append(None)
    df = pl.DataFrame({'PST': pst_list, **data_lists})
    df = df.filter(pl.col('PST').is_not_null() & (pl.col('PST').str.strip_chars() != 'PST'))
    rename_map = {old: new for old, new in zip(tmp_data_cols, DAY_METRIC_COLS[:len(tmp_data_cols)])}
    return df.rename(rename_map)

def load_ou_file(file_path: str, sheet_config: dict, file_label: str) -> dict:
    wb      = openpyxl.load_workbook(file_path, data_only=True)
    week_dt = extract_week_start(file_path)
    print(f'\n📗 {os.path.basename(file_path)}  |  Week start: {week_dt}')
    print(f'   Sheets: {wb.sheetnames}')
    result = {}
    for target, (col_s, col_e) in sheet_config.items():
        actual = _resolve_sheet(wb.sheetnames, target)
        if actual is None:
            print(f'   ⚠️  Sheet not found: "{target}" — skipped')
            continue
        df = read_ou_sheet(wb[actual], col_s, col_e).with_columns([
            pl.lit(week_dt).alias('Week'),
            pl.lit(file_label).alias('LOB'),
            pl.lit(_map_site(actual)).alias('Site'),
        ])
        result[actual] = df
        print(f'   ✅ "{actual}" → {df.shape}')
    wb.close()
    return result

# ── Load OU Mail ───────────────────────────────────────────────────────────────
lc_path = find_latest_file(folder_paths['input_ou_mail'], 'Global OU LC Chat')
nl_path = find_latest_file(folder_paths['input_ou_mail'], 'Global OU NL Chat')

lc_data = load_ou_file(lc_path, LC_SHEET_CONFIG, file_label='Lodging chat')
nl_data = load_ou_file(nl_path, NL_SHEET_CONFIG, file_label='Non-Lodging chat')

# ── Combine & Unpivot ──────────────────────────────────────────────────────────
all_combined = pl.concat([*lc_data.values(), *nl_data.values()], how='vertical')
id_cols  = ['Site', 'LOB', 'Week', 'PST']
val_cols = [c for c in all_combined.columns if c not in id_cols]

df_long = (
    all_combined
    .unpivot(index=id_cols, on=val_cols, variable_name='_col', value_name='Value')
    .with_columns([
        pl.col('_col').str.split_exact('_', 1).struct.field('field_0').alias('Day'),
        pl.col('_col').str.split_exact('_', 1).struct.field('field_1').alias('OU Status'),
    ])
    .with_columns(pl.col('Day').replace_strict(DAY_OFFSET, return_dtype=pl.Int32).alias('_offset'))
    .with_columns(
        (pl.col('Week').cast(pl.Datetime('us')) + pl.duration(days=pl.col('_offset')))
        .dt.date().alias('PST Date')
    )
    .with_columns(
        pl.concat_str([pl.col('PST Date').cast(pl.String), pl.lit(' '), pl.col('PST')])
        .str.to_datetime('%Y-%m-%d %H:%M').alias('PST Datetime')
    )
)

for tz, td in TZ_OFFSETS.items():
    us = int(td.total_seconds() * 1_000_000)
    df_long = df_long.with_columns(
        (pl.col('PST Datetime') + pl.duration(microseconds=us)).alias(f'{tz} Datetime')
    ).with_columns(
        pl.col(f'{tz} Datetime').dt.date().alias(f'{tz} Date')
    )

df_long = (
    df_long
    .with_columns([
        pl.col('Value').round(2),
        pl.col('VNT Datetime').dt.strftime('%H:%M').alias('VNT'),
    ])
    .drop(['_col', 'Day', '_offset'])
    .select([
        'Site', 'LOB', 'Week',
        'PST Date', 'PST', 'PST Datetime',
        'VNT Datetime', 'VNT Date', 'VNT',
        'IST Datetime', 'IST Date',
        'CLT Datetime', 'CLT Date',
        'OU Status', 'Value',
    ])
)

# ── Export df_long ─────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(output_ou_mail), exist_ok=True)
df_long.write_excel(output_ou_mail)

# ── IC_HCM_Details_Log ─────────────────────────────────────────────────────────
IEX_Intervals_Input = input_data(folder_paths['iex_intervals_output']).with_columns([
    pl.col('OracleID').cast(pl.Int64), pl.col('IEX ID').cast(pl.Int64),
    pl.col('Week_Monday').str.to_date('%Y-%m-%d'),
    pl.col('Date_Converted').str.to_date('%Y-%m-%d'),
    pl.col('VNT_Intervals').str.to_datetime('%Y-%m-%d %H:%M:%S'),
    pl.col('PST_Intervals').str.to_datetime('%Y-%m-%d %H:%M:%S'),
    pl.col('Datetime_Start_Time').str.to_datetime('%Y-%m-%d %H:%M:%S'),
    pl.col('Datetime_End_Time').str.to_datetime('%Y-%m-%d %H:%M:%S'),
    pl.col('Duration').cast(pl.Float64)
]).filter(pl.col('Date_Converted').dt.year() == 2026)

HC_EXTEND_COMBINED = input_data(folder_paths['hc_extend_by_month']).filter(pl.col('Year') == '2026').select([
    pl.col('Date').str.to_date('%Y-%m-%d'),
    (pl.col('Month') + '-01').str.to_date('%b-%y-%d').dt.strftime('%y_%m').alias('Month'),
    pl.col('Email Id'), pl.col('Alias'), pl.col('Designation'),
    pl.col('Supervisor Name'), pl.col('LOB'), pl.col('Active'), 'Wave'
])

IC_HCM_Details_Log = (
    IEX_Intervals_Input
    .join(HC_EXTEND_COMBINED.select(['Date', 'Email Id', 'LOB']),
          left_on=['Date_Converted', 'Email Id'], right_on=['Date', 'Email Id'], how='left')
    .with_columns(pl.col('LOB').replace_strict(LOB_MAPPING, default=pl.col('LOB')))
    .group_by(['LOB', 'VNT_Intervals', 'PST_Intervals']).agg([
        (pl.col('Duration').filter(pl.col('Scheduled Activity').is_in(['Open Time', 'Extra Hours'])).sum() * 2).alias('Scheduled_Open_Time'),
        (pl.col('Duration').filter(pl.col('Scheduled Activity').str.contains('Break')).sum() * 2).alias('Scheduled_Break'),
        (pl.col('Duration').filter(pl.col('Scheduled Activity').str.contains('Lunch')).sum() * 2).alias('Scheduled_Lunch'),
        (pl.col('Duration').filter(pl.col('Scheduled Activity').str.contains('Training|Coaching')).sum() * 2).alias('Scheduled_Training/Coaching'),
        (pl.col('Duration').filter(pl.col('Scheduled Activity').is_in(LEAVE_LIST)).sum() * 2).alias('Scheduled_Leave'),
        (pl.col('Duration').filter(pl.col('Scheduled Activity') == 'No Call/No Show').sum() * 2).alias('Scheduled_NCNS'),
        (pl.col('Duration').filter(pl.col('Scheduled Activity') == 'Termination').sum() * 2).alias('Scheduled_Terminated'),
    ])
    .with_columns([
        pl.col('PST_Intervals').dt.strftime('%Y-%m').alias('PST_Month'),
        pl.col('PST_Intervals').dt.date().alias('PST_Date'),
        pl.concat_str([pl.col('PST_Intervals').dt.strftime('%H:%M'), pl.lit('-'),
                       (pl.col('PST_Intervals') + pl.duration(minutes=30)).dt.strftime('%H:%M')]).alias('PST_Interval_Range'),
        pl.col('VNT_Intervals').dt.date().alias('VNT_Date'),
        pl.concat_str([pl.col('VNT_Intervals').dt.strftime('%H:%M'), pl.lit('-'),
                       (pl.col('VNT_Intervals') + pl.duration(minutes=30)).dt.strftime('%H:%M')]).alias('VNT_Interval_Range'),
    ])
    .select([
        'LOB', 'PST_Month', 'PST_Date', 'PST_Intervals', 'PST_Interval_Range',
        'VNT_Date', 'VNT_Intervals', 'VNT_Interval_Range',
        'Scheduled_Open_Time', 'Scheduled_Break', 'Scheduled_Lunch',
        'Scheduled_Training/Coaching', 'Scheduled_Leave', 'Scheduled_NCNS',
        'Scheduled_Terminated',
    ])
)

print(f'✅ IC_HCM_Details_Log : {IC_HCM_Details_Log.shape}')

# ── df_hcm_wide (HCM + Global) ────────────────────────────────────────────────
df_hcm_wide = (
    df_long
    .filter(pl.col('Site') == 'Concentrix (Ho Chi Minh City)')
    .select(['LOB', 'PST Datetime', 'OU Status', 'Value'])
    .pivot(on='OU Status', index=['LOB', 'PST Datetime'], values='Value')
    .rename({'Req W': 'Req', 'DIFF': 'Diff'})
)

df_global_wide = (
    df_long
    .filter(pl.col('Site') == 'Concentrix (Global)')
    .select(['LOB', 'PST Datetime', 'OU Status', 'Value'])
    .pivot(on='OU Status', index=['LOB', 'PST Datetime'], values='Value')
    .rename({'Req W': 'Global_Req', 'Prov': 'Global_Prov', 'DIFF': 'Global_Diff'})
)

df_hcm_wide = df_hcm_wide.join(
    df_global_wide, on=['LOB', 'PST Datetime'], how='left'
)

# ── Final Merge ────────────────────────────────────────────────────────────────
df_merged = df_hcm_wide.join(
    IC_HCM_Details_Log,
    left_on  = ['LOB', 'PST Datetime'],
    right_on = ['LOB', 'PST_Intervals'],
    how      = 'left'
).rename({
    'Req' : 'VN_Req',
    'Prov': 'VN_Prov',
    'Diff': 'VN_Diff',
}).select([
    'LOB', 'PST Datetime',
    'PST_Month', 'PST_Date', 'PST_Interval_Range',
    'VNT_Date', 'VNT_Intervals', 'VNT_Interval_Range',
    'VN_Req', 'VN_Prov', 'VN_Diff',
    'Global_Req', 'Global_Prov', 'Global_Diff',
    'Scheduled_Open_Time', 'Scheduled_Break', 'Scheduled_Lunch',
    'Scheduled_Training/Coaching', 'Scheduled_Leave', 'Scheduled_NCNS',
    'Scheduled_Terminated',
])

print(f'\ndf_long     : {df_long.shape}')
print(f'df_hcm_wide : {df_hcm_wide.shape}')
print(f'df_merged   : {df_merged.shape}')
display(df_merged.head(10))

c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)



📗 Global OU LC Chat 07_20_26.xlsx  |  Week start: 2026-07-20
   Sheets: ['LookingGlass Template', 'Client View', 'Forecast var.', 'Week 1 - OU', 'Weekly Staffing - Week 1', 'Email Template - Week 1', 'Performance Analysis', 'Egypt - OU', 'Egypt Staffing - Data', 'VNM - OU ', 'VNM Staffing - Data', 'Kol - OU ', 'Kol Staffing - Data', 'Pune - OU', 'Pune Staffing - Data', 'Gobal - OU', 'Volume']
   ✅ "VNM - OU " → (48, 25)
   ✅ "Kol - OU " → (48, 25)
   ✅ "Pune - OU" → (48, 25)
   ✅ "Gobal - OU" → (48, 25)

📗 Global OU NL Chat 07_20_26.xlsx  |  Week start: 2026-07-20
   Sheets: ['LookingGlass Template', 'Client View', 'Forecast var.', 'Week 1 - OU', 'Weekly Staffing - Week 1', 'Email Template - Week 1', 'Performance Analysis', 'Egypt - OU', 'VNM - OU ', 'VNM Staffing - Data', 'Egypt Staffing - Data', 'Kol Staffing - Data', 'Pune - OU', 'Sheet1', 'Pune Staffing - Data', 'Kol - OU ', 'Gobal - OU', 'Requirement Trend', 'Req+Open', 'Volume', 'Shinkage', 'Net Staffing']
   ✅ "VNM - OU " → (48

LOB,PST Datetime,PST_Month,PST_Date,PST_Interval_Range,VNT_Date,VNT_Intervals,VNT_Interval_Range,VN_Req,VN_Prov,VN_Diff,Global_Req,Global_Prov,Global_Diff,Scheduled_Open_Time,Scheduled_Break,Scheduled_Lunch,Scheduled_Training/Coaching,Scheduled_Leave,Scheduled_NCNS,Scheduled_Terminated
str,datetime[μs],str,date,str,date,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lodging chat""",2026-07-20 00:00:00,"""2026-07""",2026-07-20,"""00:00-00:30""",2026-07-20,2026-07-20 14:00:00,"""14:00-14:30""",11.4,16.89,5.49,19.29,29.58,10.29,19.5,0.0,2.5,1.0,2.0,0.0,0.0
"""Lodging chat""",2026-07-20 00:30:00,"""2026-07""",2026-07-20,"""00:30-01:00""",2026-07-20,2026-07-20 14:30:00,"""14:30-15:00""",11.73,18.18,6.45,18.48,29.54,11.06,21.0,0.0,2.0,0.0,2.0,0.0,0.0
"""Lodging chat""",2026-07-20 01:00:00,"""2026-07""",2026-07-20,"""01:00-01:30""",2026-07-20,2026-07-20 15:00:00,"""15:00-15:30""",8.26,8.52,0.26,19.66,21.35,1.69,9.833333,0.0,1.166667,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 01:30:00,"""2026-07""",2026-07-20,"""01:30-02:00""",2026-07-20,2026-07-20 15:30:00,"""15:30-16:00""",7.4,7.79,0.39,18.45,20.49,2.04,9.0,0.0,2.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 02:00:00,"""2026-07""",2026-07-20,"""02:00-02:30""",2026-07-20,2026-07-20 16:00:00,"""16:00-16:30""",7.23,7.79,0.56,19.66,22.36,2.7,9.0,0.0,2.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 02:30:00,"""2026-07""",2026-07-20,"""02:30-03:00""",2026-07-20,2026-07-20 16:30:00,"""16:30-17:00""",6.15,6.06,-0.09,20.35,21.29,0.94,7.0,0.0,4.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 03:00:00,"""2026-07""",2026-07-20,"""03:00-03:30""",2026-07-20,2026-07-20 17:00:00,"""17:00-17:30""",5.51,5.34,-0.17,19.21,19.77,0.56,6.166667,1.0,3.833333,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 03:30:00,"""2026-07""",2026-07-20,"""03:30-04:00""",2026-07-20,2026-07-20 17:30:00,"""17:30-18:00""",9.79,9.09,-0.7,23.25,22.72,-0.53,11.5,0.5,0.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 04:00:00,"""2026-07""",2026-07-20,"""04:00-04:30""",2026-07-20,2026-07-20 18:00:00,"""18:00-18:30""",4.21,7.79,3.58,10.76,21.02,10.26,10.0,2.0,0.0,0.0,1.0,0.0,0.0


In [3]:
# ── df_hcm_wide (HCM + Global) ─────────────────────────────────────────────────
df_hcm_wide = (
    df_long
    .filter(pl.col('Site') == 'Concentrix (Ho Chi Minh City)')
    .select(['LOB', 'PST Datetime', 'OU Status', 'Value'])
    .pivot(on='OU Status', index=['LOB', 'PST Datetime'], values='Value')
    .rename({'Req W': 'Req', 'DIFF': 'Diff'})
)

df_global_wide = (
    df_long
    .filter(pl.col('Site') == 'Concentrix (Global)')
    .select(['LOB', 'PST Datetime', 'OU Status', 'Value'])
    .pivot(on='OU Status', index=['LOB', 'PST Datetime'], values='Value')
    .rename({'Req W': 'Global_Req', 'Prov': 'Global_Prov', 'DIFF': 'Global_Diff'})
)

df_hcm_wide = df_hcm_wide.join(
    df_global_wide, on=['LOB', 'PST Datetime'], how='left'
)

# ── Merge với IC_HCM_Details_Log ───────────────────────────────────────────────
df_merged = df_hcm_wide.join(
    IC_HCM_Details_Log,
    left_on  = ['LOB', 'PST Datetime'],
    right_on = ['LOB', 'PST_Intervals'],
    how      = 'left'
).rename({
    'Req' : 'VN_Req',
    'Prov': 'VN_Prov',
    'Diff': 'VN_Diff',
}).select([
    'LOB', 'PST Datetime',
    'PST_Month', 'PST_Date', 'PST_Interval_Range',
    'VNT_Date', 'VNT_Intervals', 'VNT_Interval_Range',
    'VN_Req', 'VN_Prov', 'VN_Diff',
    'Global_Req', 'Global_Prov', 'Global_Diff',
    'Scheduled_Open_Time', 'Scheduled_Break', 'Scheduled_Lunch',
    'Scheduled_Training/Coaching', 'Scheduled_Leave', 'Scheduled_NCNS',
    'Scheduled_Terminated',
])

print(f'Shape   : {df_merged.shape}')
print(f'Columns : {df_merged.columns}')
display(df_merged.head(10))

Shape   : (672, 21)
Columns : ['LOB', 'PST Datetime', 'PST_Month', 'PST_Date', 'PST_Interval_Range', 'VNT_Date', 'VNT_Intervals', 'VNT_Interval_Range', 'VN_Req', 'VN_Prov', 'VN_Diff', 'Global_Req', 'Global_Prov', 'Global_Diff', 'Scheduled_Open_Time', 'Scheduled_Break', 'Scheduled_Lunch', 'Scheduled_Training/Coaching', 'Scheduled_Leave', 'Scheduled_NCNS', 'Scheduled_Terminated']


LOB,PST Datetime,PST_Month,PST_Date,PST_Interval_Range,VNT_Date,VNT_Intervals,VNT_Interval_Range,VN_Req,VN_Prov,VN_Diff,Global_Req,Global_Prov,Global_Diff,Scheduled_Open_Time,Scheduled_Break,Scheduled_Lunch,Scheduled_Training/Coaching,Scheduled_Leave,Scheduled_NCNS,Scheduled_Terminated
str,datetime[μs],str,date,str,date,datetime[μs],str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Lodging chat""",2026-07-20 00:00:00,"""2026-07""",2026-07-20,"""00:00-00:30""",2026-07-20,2026-07-20 14:00:00,"""14:00-14:30""",11.4,16.89,5.49,19.29,29.58,10.29,19.5,0.0,2.5,1.0,2.0,0.0,0.0
"""Lodging chat""",2026-07-20 00:30:00,"""2026-07""",2026-07-20,"""00:30-01:00""",2026-07-20,2026-07-20 14:30:00,"""14:30-15:00""",11.73,18.18,6.45,18.48,29.54,11.06,21.0,0.0,2.0,0.0,2.0,0.0,0.0
"""Lodging chat""",2026-07-20 01:00:00,"""2026-07""",2026-07-20,"""01:00-01:30""",2026-07-20,2026-07-20 15:00:00,"""15:00-15:30""",8.26,8.52,0.26,19.66,21.35,1.69,9.833333,0.0,1.166667,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 01:30:00,"""2026-07""",2026-07-20,"""01:30-02:00""",2026-07-20,2026-07-20 15:30:00,"""15:30-16:00""",7.4,7.79,0.39,18.45,20.49,2.04,9.0,0.0,2.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 02:00:00,"""2026-07""",2026-07-20,"""02:00-02:30""",2026-07-20,2026-07-20 16:00:00,"""16:00-16:30""",7.23,7.79,0.56,19.66,22.36,2.7,9.0,0.0,2.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 02:30:00,"""2026-07""",2026-07-20,"""02:30-03:00""",2026-07-20,2026-07-20 16:30:00,"""16:30-17:00""",6.15,6.06,-0.09,20.35,21.29,0.94,7.0,0.0,4.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 03:00:00,"""2026-07""",2026-07-20,"""03:00-03:30""",2026-07-20,2026-07-20 17:00:00,"""17:00-17:30""",5.51,5.34,-0.17,19.21,19.77,0.56,6.166667,1.0,3.833333,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 03:30:00,"""2026-07""",2026-07-20,"""03:30-04:00""",2026-07-20,2026-07-20 17:30:00,"""17:30-18:00""",9.79,9.09,-0.7,23.25,22.72,-0.53,11.5,0.5,0.0,0.0,1.0,0.0,0.0
"""Lodging chat""",2026-07-20 04:00:00,"""2026-07""",2026-07-20,"""04:00-04:30""",2026-07-20,2026-07-20 18:00:00,"""18:00-18:30""",4.21,7.79,3.58,10.76,21.02,10.26,10.0,2.0,0.0,0.0,1.0,0.0,0.0
